# KV Cache缓存

以LLaMA-7B
- 层数 L = 32
- 注意力头数 h = 32
- 每头的维度 d_h = 128
- 总隐藏维度 d_model = 4096

### 单步缓存大小：

In [2]:
import torch

# 每层每个token的K/V缓存

batch_size = 1
num_heads = 32
head_dim = 128
seq_len = 1

# key cache : [batch,num_heads,seq_len,head_dim]
k_cache_shape = (1,32,1,128)
k_cache = torch.zeros(k_cache_shape) # 4KB per token per layer

# value cache 形状相同
v_cache_shape =  (1,32,1,128)
v_cache = torch.zeros(v_cache_shape) # 4KB

# 单层 K+V = 8KB
# 32层的 k+v = 32 * 8 = 256KB

print(v_cache.shape)

torch.Size([1, 32, 1, 128])


### 完整缓存 （序列长度2048）


In [3]:
max_seq_len = 2048
num_layers = 32

# 预分配最大缓存
k_cache = torch.zeros(batch_size,num_layers,num_heads,max_seq_len,head_dim)
v_cache = torch.zeros(batch_size,num_layers,num_heads,max_seq_len,head_dim)

# 显存占用： 2 *  32 * 2048 * 128 * 2 bytes(fp16) = 32MB
# 实际使用中 batch_size = 8 , 8*32 = 256MB

In [ ]:
用户输入："今天天气怎么样？"
                    ↓
            ┌───────────────┐
            │  预填充阶段    │  ← 一次性处理整个输入
            │  (Prefill)    │
            └───────────────┘
                    ↓
            生成第一个token："今天"
                    ↓
            ┌───────────────┐
            │   解码阶段    │  ← 逐个生成后续token
            │   (Decode)    │     循环直到结束
            └───────────────┘
                    ↓
            完整输出："今天天气很好，适合出门。"

## 预填充阶段

一次性处理用户的完整输入（Prompt），填充初始 KV Cache。建立初始上下文

关键特点：
-  并行计算：一次性处理所有输入token
-  因果掩码：每个位置只能看到前面的token（包括自己）
-  全量缓存：所有位置的 K/V 都被保存
-  计算密集：这是推理中最耗时的阶段（但只做一次）

In [ ]:
输入：["今", "天", "天", "气", "怎", "么", "样", "？"] (8个token)
        ↓
    并行计算所有位置的 Q, K, V
        ↓
    保存 K, V 到缓存（供后续解码使用）
        ↓
    输出下一个token的预测："今天"

In [ ]:
Layer 0 的 KV Cache 状态（预填充后）:
┌────────────────────────────────────────────────────────────┐
│ K Cache: [layer=0, batch=0, kv_heads=8, seq=2048, dim=128] │
│                                                            │
│ Head 0:  [k₀, k₁, k₂, k₃, k₄, k₅, k₆, k₇, _, _, ...]       │
│          ↑  ↑  ↑  ↑  ↑  ↑  ↑  ↑                            │
│          │  │  │  │  │  │  │  └ "？"的位置编码+向量        │
│          │  │  │  │  │  │  └ "么"的位置编码+向量           │
│          │  │  │  │  │  └ "怎"的位置编码+向量              │
│          │  │  │  │  └ "气"的位置编码+向量                 │
│          │  │  │  └ "天"(第2个)的位置编码+向量             │
│          │  │  └ "天"(第1个)的位置编码+向量（不同位置编码！）│
│          │  └ "今"的位置编码+向量                          │
│          └ [PAD] 或起始标记                               │
│                                                            │
│ V Cache: 相同结构，存储Value向量                           │
│                                                            │
│ 当前缓存长度: cache_len = 8                                │
│ 下一个写入位置: position = 8                               │
└────────────────────────────────────────────────────────────┘

In [1]:
import torch

batch_size = 2   # 批量大小（同时处理2个句子）
seq_len = 4      # 序列长度（每个句子4个token）
hidden_dim = 4096 # 隐藏层维度

# 经过 q_proj 线性层后的输出
q = torch.randn(batch_size, seq_len, hidden_dim)
print(f"原始形状: {q.shape}")  # [2, 4, 4096]

原始形状: torch.Size([2, 4, 4096])


In [ ]:
q: [2, 4, 4096] = [batch_size, seq_len, hidden_dim]

Batch 0:  [ [q₀₀₀, q₀₀₁, q₀₀₂, ..., q₀₄₀₉₅],   ← token 0 的 4096 维向量
           [q₀₁₀, q₀₁₁, q₀₁₂, ..., q₀₁₄₀₉₅],   ← token 1 的 4096 维向量
           [q₀₂₀, q₀₂₁, q₀₂₂, ..., q₀₂₄₀₉₅],   ← token 2
           [q₀₃₀, q₀₃₁, q₀₃₂, ..., q₀₃₄₀₉₅] ]  ← token 3

Batch 1:  [ [q₁₀₀, q₁₀₁, q₁₀₂, ..., q₁₀₄₀₉₅],   ← 第2个句子的 token 0
           [q₁₁₀, q₁₁₁, q₁₁₂, ..., q₁₁₄₀₉₅],
           ... ]

In [ ]:
q = q.view(batch_size, seq_len, 32, 128)
print(f"view后形状: {q.shape}")  # [2, 4, 32, 128]

In [ ]:
4096 = 32 × 128
      ↑    ↑
    头数  每头维度
   (num_heads) (head_dim)

In [ ]:
# 内存变化

原始: [2, 4, 4096]  →  总共 2×4×4096 = 32,768 个数

view: [2, 4, 32, 128] → 还是 2×4×32×128 = 32,768 个数
                              ↑
                        只是重新解释内存布局！

In [ ]:
q: [2, 4, 32, 128] = [batch, seq, heads, head_dim]

Batch 0, Token 0:  [ [h₀₀₀...h₀₁₂₇],   ← 第0头的128维
                     [h₁₀₀...h₁₁₂₇],   ← 第1头的128维
                     ...
                     [h₃₁₀₀...h₃₁₁₂₇] ]  ← 第31头的128维

每个token的4096维被拆成32个128维的向量！

In [ ]:
# 转置后

q: [2, 32, 4, 128] = [batch, heads, seq, head_dim]

Batch 0:
  Head 0:  [ [h₀₀₀...h₀₁₂₇],   ← token 0
             [h₀₁₀...h₀₁₂₇],   ← token 1
             [h₀₂₀...h₀₂₂₇],   ← token 2
             [h₀₃₀...h₀₃₂₇] ]  ← token 3
           ↑
        这32个数在内存中连续！

  Head 1:  [ [h₁₀₀...h₁₁₂₇],
             [h₁₁₀...h₁₁₂₇],
             [h₁₂₀...h₁₂₂₇],
             [h₁₃₀...h₁₃₂₇] ]
  
  ...

  Head 31: [ [h₃₁₀₀...h₃₁₁₂₇],
             ... ]
    
    
# 转置后的内存布局（理想）：
# [batch 0, head 0, token 0-3, dim] 连续存储
# [batch 0, head 1, token 0-3, dim] 连续存储
# ...

# 这样 PyTorch 可以用高效的 BLAS 库并行计算32个头的矩阵乘法

### 逐头写入

In [ ]:
# 当前计算的 k: [1, 8, 8, 128]
# 包含8个token，8个KV头，每个128维

kv_cache.k[5, :, :, :8] = k

In [ ]:
KV Head 0:
k[0, 0, :, :] 的形状是 [8, 128]
            ↓
kv_cache.k[5, 0, 0, 0:8, :] = k[0, 0, :, :]

写入后:
[ [k₀₀₀...k₀₁₂₇],   ← pos 0: token "今" 的 K
  [k₀₁₀...k₀₁₂₇],   ← pos 1: token "天" 的 K
  [k₀₂₀...k₀₂₂₇],   ← pos 2: token "天" 的 K
  [k₀₃₀...k₀₃₂₇],   ← pos 3: token "气" 的 K
  [k₀₄₀...k₀₄₂₇],   ← pos 4: token "怎" 的 K
  [k₀₅₀...k₀₅₂₇],   ← pos 5: token "么" 的 K
  [k₀₆₀...k₀₆₂₇],   ← pos 6: token "样" 的 K
  [k₀₇₀...k₀₇₂₇],   ← pos 7: token "？" 的 K
  [ _, _, _, _ ],   ← pos 8: 空
  ...               ← pos 9~2047: 空
]

KV Head 1~7: 同样方式写入对应数据

### 写入后状态



In [ ]:
Layer 5, Batch 0:
┌────────────────────────────────────────────────────────────┐
│ KV Head 0:                                                 │
│   [■■■■■■■■, _, _, _, ... , _ ]                            │
│    ↑↑↑↑↑↑↑↑                                                │
│   8个位置已填充，其余2040个位置预留                           │
│                                                            │
│ KV Head 1:  [■■■■■■■■, _, _, _, ... , _ ]                   │
│ ...                                                        │
│ KV Head 7:  [■■■■■■■■, _, _, _, ... , _ ]                   │
│                                                            │
│ 图例: ■ = 已填充, _ = 空                                   │
└────────────────────────────────────────────────────────────┘

In [ ]:
def prefill_phase(prompt_tokens,model,kv_cache):
    '''
    prompt_tokens : [batch_size=1,seq_len=8]
                   例如: [[101, 234, 234, 567, 890, 123, 456, 789]]
                   token ID 列表
    '''
    batch_size,seq_len = prompt_tokens.shape # 1,8
   
    
    #  ========== 步骤1: 词嵌入 ==========
    # 将token ID转换为向量
    hidden_states = model.embed(prompt_tokens)
    # 形状：[1,8,4096] (batch,seq_len,hidden_dim)
    
    # ========== 步骤2: 逐层处理 ==========
    for layer_idx in range(model.num_layers): # 32层
        # --- 2.1 计算 Q, K, V ---
        # 线性投影：hidden -> q,k,v
        q = model.layers[layer_idx].q_proj(hidden_states) # [1,8,4096 = 32 * 128]
        k = model.layers[layer_idx].k_proj(hidden_states) # [1,8,1024]  GQA压缩
        v = model.layers[layer_idx].v_proj(hidden_states) # [1,8,1024]
        
        # 重塑为多头形式
        # q :[1,8,32,128] ->[1,32,8,128] (batch,heads,seq,dim) 4096 = 32 * 128
        q = q.view(batch_size,seq_len,32,128).transpose(1,2)
        k = k.view(batch_size,seq_len,8,128).transpose(1,2) # 8个kv头
        v = v.view(batch_size,seq_len,8,128).transpose(1,2)
        
        # --- 2.2 应用旋转位置编码 (RoPE) ---
        # 关键：给q,k添加位置信息
        q, k = apply_rotary_pos_emb(q, k, position_offset=0)
        # q/k 形状保持: [1, heads, 8, 128]
        
        # --- 2.3 ★★★ 保存到 KV Cache ★★★ ---
        # 这是预填充阶段的核心！保存所有位置的K,V
        kv_cache.k[layer_idx, :, :, :seq_len] = k  # 保存8个位置
        kv_cache.v[layer_idx, :, :, :seq_len] = v  # 保存8个位置
        
        # 缓存形状: [32, 1, 8, 2048, 128] (layers, batch, kv_heads, max_seq, dim)
        # 实际使用: [32, 1, 8, 8, 128] (前8个位置)
        
        # --- 2.4 计算自注意力 ---
        # 因果掩码: 只能看到当前及之前位置
        # 掩码矩阵: 8×8 下三角
        # [[1, 0, 0, 0, 0, 0, 0, 0],
        #  [1, 1, 0, 0, 0, 0, 0, 0],
        #  [1, 1, 1, 0, 0, 0, 0, 0],
        #  ...]
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / sqrt(128)
        # 形状: [1, 32, 8, 8] (batch, heads, q_len, kv_len)
        
        # 应用因果掩码
        causal_mask = torch.triu(torch.ones(8, 8), diagonal=1).bool()
        attn_scores = attn_scores.masked_fill(causal_mask, float('-inf'))
        
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v)  # [1, 32, 8, 128]
        
        # --- 2.5 后续处理 ---
        # 重塑 + 线性投影 + FFN
        attn_output = attn_output.transpose(1, 2).contiguous().view(1, 8, 4096)
        hidden_states = model.layers[layer_idx].ffn(attn_output)
    
    # ========== 步骤3: 输出层 ==========
    # 只取最后一个位置的 hidden state 预测下一个token
    last_hidden = hidden_states[:, -1, :]  # [1, 4096]
    logits = model.lm_head(last_hidden)     # [1, vocab_size=50000]
    
    # 采样生成第一个token
    next_token = sampling(logits)  # 例如: token_id = 456 ("今天")
    
    return next_token, kv_cache  # 返回token和填充好的缓存

## 解码阶段（Decode Phase）

自回归生成：每次只输入一个新token，利用缓存的历史 K/V 加速

In [ ]:
第1步解码:
输入: ["今天"] (1个token，不是8个！)
       ↓
   读取缓存中8个历史K/V + 计算新K/V
       ↓
   Attention: Q_new × [K_history, K_new]
       ↓
   生成: "天气"

第2步解码:
输入: ["天气"] (1个token)
       ↓
   读取缓存中9个历史K/V + 计算新K/V
       ↓
   生成: "很"

...重复直到生成结束标记

#### 解码阶段的可视化

In [ ]:
第1步解码（position=8，生成"天气"）:
┌─────────────────────────────────────────────────────────────┐
│ 输入: ["今天"] (token_id=456)                                │
│                                                             │
│ Layer 0 处理:                                                │
│   Q: 只计算 "今天" 的query [1, 32, 1, 128]                   │
│                                                             │
│   K/V Cache 读取:                                            │
│   [k₀, k₁, k₂, k₃, k₄, k₅, k₆, k₇]  历史8个位置              │
│      ↓ 拼接                                                  │
│   [k₀, k₁, k₂, k₃, k₄, k₅, k₆, k₇, k₈]  加入新位置8          │
│                                    ↑                        │
│                              "今天"的K（位置编码=8）         │
│                                                             │
│   Attention: Q_new × K_full^T                                 │
│   [1, 32, 1, 128] × [1, 8, 128, 9] = [1, 32, 1, 9]          │
│                    ↑                                        │
│              看到全部9个位置（含自己）                        │
│                                                             │
│   输出: "天气"                                               │
│   Cache更新: position = 9                                    │
└─────────────────────────────────────────────────────────────┘

第2步解码（position=9，生成"很"）:
┌─────────────────────────────────────────────────────────────┐
│ 输入: ["天气"] (token_id=789)                                │
│                                                             │
│   K/V Cache 现在包含9个历史位置                               │
│   [k₀...k₈] + k₉("天气")                                     │
│                                                             │
│   Attention 计算: [1, 32, 1, 128] × [1, 8, 128, 10]        │
│                                                             │
│   输出: "很"                                                 │
│   Cache更新: position = 10                                   │
└─────────────────────────────────────────────────────────────┘

### 关键优化点

In [ ]:
# 优化1: Q长度始终为1，计算量 O(n) 而非 O(n²)
attn_scores = torch.matmul(q, k_full.transpose(-2, -1))
# q: [..., 1, dim], k_full: [..., n, dim]
# 结果: [..., 1, n]  只需计算一行！

# 对比预填充:
# q: [..., n, dim], k: [..., n, dim]
# 结果: [..., n, n]  需要计算完整矩阵

# 优化2: 无需重复计算历史K/V
k_history = kv_cache.k[layer_idx, :, :, :position]  # O(1) 读取
# 对比无缓存: 需要重新计算所有历史token的K/V

# 优化3: 内存访问连续
# 缓存是连续内存，读取速度快

In [ ]:
def decode_phase(last_token, model, kv_cache, position):
    '''
    last_token:[batch_size=1,seq_len=1] 例如[[456]]('今天')
    postion: int = 8 （当前写入位置，即历史长度。追加）
    '''
    batch_size = 1
    new_len =1 # 每次只处理1个新token
    
    # ========== 步骤1: 只嵌入新token ==========
    hidden = model.embed(last_token) # [1,1,4096]
    # 对比预填充：[1,8,4096]
    
    # ========== 步骤2: 逐层处理 ==========
    for layer_idx in range(model.num_layers):
        # --- 2.1 只计算新token的 Q, K, V ---
        q = model.layers[layer_idx].q_proj(hidden)     # [1,1,4096] 4096 = 32头 * 128维
        k_new = model.layers[layer_idx].k_proj(hidden) # [1,1,1024]
        v_new = model.layers[layer_idx].v_proj(hidden) # [1,1,1024] 1024 = 8头 * 128维
        
        # 重塑
        q = q.view(1,1,32,128).transpose(1,2) # [1,32,1,128]
        k_new = k_new.view(1,1,8,128).transpose(1,2) # [1,8,1,128]
        v_new = v_new.view(1,1,8,128).transpose(1,2) # [1,8,1,128]
        
        # --- 2.2 位置编码（关键差异！）---
        # 新token的位置 = 当前position（不是0！）
        q, k_new = apply_rotary_pos_emb(q, k_new, position_offset=position)
        # 例如: position=8，使用第8个位置的角度编码
        
        # --- 2.3 ★★★ 读取历史 + 拼接 ★★★ ---
        # 从缓存读取 [0:position] 的历史K/V
        
        # 从缓存中读取[0:postion]的历史K/V
        k_history = kv_cache.k[layer_idx,:,:,:postion] # [1,8,8,128]
        v_history = kv_cache.v[layer_idx,:,:,:postion] # [1,8,8,128]
        # 注意这里读取的是与填充阶段保存的8个位置
        
        # 拼接历史和新K/V
        k_full = torch.cat([k_history,k_new],dim=2) # [1,8,9,128] 
        v_full = torch.cat([v_history,v_new],dim=2) # [1,8,9,128]
        # 形状 [batch,kv_heads,history_len+new_len,dim]
        
        # --- 2.4 ★★★ 更新缓存 ★★★ ---
        # 将新K/V保存到position位置
        kv_cache.k[layer_idx,:,:,postion] = k_new.squeeze(2) # [1,8,1,128]->[1,8,128]
        kv_cache.v[layer_idx,:,:,postion] = v_new.squeeze(2) # [1,8,128]
        
        # 缓存现在包含9个位置
        
        #  --- 2.5 计算注意力（关键差异！）---
        # Q: 只有新token的query [1,32,1,128]
        # K/V: 完整历史 + 新token [1,8,9,128]
        
        # 计算注意力分数
        atten_scores = torch.matmul(q,k_full.transpose(-2,-1)) / sqrt(128)
        # 形状： [1,32,1,9] (batch，heads,q_len=1,kv_len=9) 这里的Q长度是1
        # 对于预填充：[1,32,8,8] 
        
        # 无需因果掩码（或简化为检查）
        # 因为Q只有一个位置，自然只能看到所有历史
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v_full)  # [1, 32, 1, 128]
        
        # --- 2.6 后续处理 ---
        attn_output = attn_output.transpose(1, 2).view(1, 1, 4096)
        hidden = model.layers[layer_idx].ffn(attn_output)
        # 输出: [1, 1, 4096]
        
    # ========== 步骤3: 生成下一个token ==========
    logits = model.lm_head(hidden.squeeze(1))  # [1, 50000]
    next_token = sampling(logits)  # 例如: "天气"
    
    return next_token,kv_cache # 返回新token和跟新后的缓存

### 两阶段对比总结

| 维度               | 预填充阶段 (Prefill)         | 解码阶段 (Decode) |
| ---------------- | ----------------------- | ------------- |
| **输入长度**         | 长（整个prompt，如512 tokens） | 短（1个新token）   |
| **计算方式**         | 并行（矩阵乘法）                | 增量（向量操作）      |
| **KV Cache 操作**  | 写入（填充初始值）               | 读取历史 + 写入新值   |
| **Attention 掩码** | 需要因果掩码（下三角）             | 无需掩码（Q长度=1）   |
| **计算复杂度**        | O(n²)                   | O(n)          |
| **内存带宽**         | 计算密集                    | 内存密集（读取缓存）    |
| **执行次数**         | 1次                      | n次（自回归循环）     |
| **性能瓶颈**         | 矩阵乘法速度                  | 缓存读取带宽        |


## 完整调用流程

In [2]:
def generate(model,tokenizer,prompt,max_new_tokens = 100):
    # 1 编码输入
    input_ids = tokenizer.encode(prompt) # [101, 234, 567, ...]
    
    # 2. 初始化 KV Cache
    kv_cache = KVCache(
        num_layers = 32,
        num_heads = 32,
        head_dim = 128,
        max_seq_len = 2048
    )
    
    # ========== 预填充阶段（只执行1次）==========
    print("=== Prefill Phase ===")
    next_token,kv_cache = prefill_phase(
        input_ids.unsqueeze(0), # [prompt_len] ->[1,prompt_len]
        model,
        kv_cache,
    )
    generated = [next_token]
    postion = len(input_ids) # 8
    
    # ========== 解码阶段（循环执行）==========
    print("=== Decode Phase ===")
    for i in range(max_new_tokens):
        # 每次只传1个token
        next_token,kv_cache = decode_phase(
            next_token.unsqueeze(0), # [1,1]
            model,
            kv_cache,
            postion,
        )
        # append 进generated
        generated.append(next_token)
        postion +=1
        
        # 结束
        if next_token == tokenizer.eos_token_id:
            break
    return tokenizer.decode(generated)

### 缓存满了怎么办？

In [ ]:
# 方案1 ：滑动窗口，（丢弃最老的）
if postion >= max_seq_len:
    # 丢弃前一半，保留后一半
    kv_cache.k = kv_cache.k[:,:,:,max_seq_len // 2:]
    postion = max_seq_len // 2
    

# 方案2 ：分页缓存（vLLM）
# 动态分配物理块，支持超长序列

# 方案3：压缩、量化
# 将老缓存压缩到更低精度